# ANN Salary Predictor - PyTorch Version
This notebook contains the PyTorch implementation of an Artificial Neural Network (ANN) for salary prediction.
The model uses dense layers with dropout regularization to predict job salaries based on various features.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import json

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load the dataset
data = pd.read_csv('/kaggle/input/datasets/vrajesh0sharma7/salary-prediction-250k/job_salary.csv')

print(f"Dataset shape: {data.shape}")
print(f"\nFirst few samples:")
print(data.head(10))
print(f"\nDataset info:")
print(data.info())

In [ ]:
# Check unique job titles
print(f"Number of unique job titles: {len(data.job_title.unique())}")
print(f"\nSample job titles: {data.job_title.unique()[:10]}")

In [ ]:
# Preprocess the data
# One-hot encode categorical features
data = pd.get_dummies(
    data, 
    columns=['job_title', 'industry', 'location', 'remote_work'],
    dtype='int'
)

print(f"Data shape after one-hot encoding: {data.shape}")
print(f"\nFirst few columns: {data.columns[:10].tolist()}")
print(f"\nData sample:")
print(data.head())

In [ ]:
# Encode ordinal categorical features using factorize
data['education_level'] = pd.factorize(data['education_level'])[0]
data['company_size'] = pd.factorize(data['company_size'])[0]

print(f"Data shape after encoding ordinal features: {data.shape}")
print(f"\nData preview:")
print(data.head())
print(f"\nData types:")
print(data.dtypes)

In [ ]:
# Standardize the data using StandardScaler
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

print(f"Scaled data shape: {data_scaled.shape}")
print(f"\nScaled data statistics:")
print(f"Mean: {data_scaled.mean():.6f}")
print(f"Std: {data_scaled.std():.6f}")
print(f"\nScaled data sample:")
print(pd.DataFrame(data_scaled).head())

In [ ]:
# Split features and target
X = pd.DataFrame(data_scaled)  # All columns except last (will be done after train-test split)
y_data = data.iloc[:, -1]  # Last column is the target (salary)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y_data.shape}")
print(f"\nFeature statistics:")
print(X.describe())

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y_data, test_size=0.2, random_state=42
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Test target shape: {y_test.shape}")

In [ ]:
# Convert to PyTorch tensors
X_train_tensor = torch.from_numpy(X_train.values).float().to(device)
y_train_tensor = torch.from_numpy(y_train.values).float().to(device)

X_test_tensor = torch.from_numpy(X_test.values).float().to(device)
y_test_tensor = torch.from_numpy(y_test.values).float().to(device)

print(f"X_train tensor shape: {X_train_tensor.shape}")
print(f"y_train tensor shape: {y_train_tensor.shape}")
print(f"X_test tensor shape: {X_test_tensor.shape}")
print(f"y_test tensor shape: {y_test_tensor.shape}")

In [ ]:
# Create data loaders
batch_size = 32

# Training data loader with validation split
validation_split = 0.2
train_size = int((1 - validation_split) * len(X_train_tensor))
val_size = len(X_train_tensor) - train_size

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

# Test data loader
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training loader batches: {len(train_loader)}")
print(f"Validation loader batches: {len(val_loader)}")
print(f"Test loader batches: {len(test_loader)}")

In [ ]:
# Define the ANN model
class SalaryPredictorANN(nn.Module):
    def __init__(self, input_size, dropout_rate=0.5):
        super(SalaryPredictorANN, self).__init__()
        
        # Dense layers with ReLU activation
        self.fc1 = nn.Linear(input_size, 64)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(64, 128)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(128, 512)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout_rate)
        
        # Output layer with linear activation (for regression)
        self.fc4 = nn.Linear(512, 1)
    
    def forward(self, x):
        # Layer 1
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        
        # Layer 2
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        
        # Layer 3
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.dropout3(x)
        
        # Output layer
        x = self.fc4(x)
        
        return x


# Initialize model
input_size = X_train_tensor.shape[1]
model = SalaryPredictorANN(input_size, dropout_rate=0.5)
model = model.to(device)

print(f"Model initialized:")
print(model)

In [ ]:
# Print model summary with parameter count
print("\n" + "="*60)
print("MODEL SUMMARY")
print("="*60)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nTotal Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"\nModel Architecture:")
print(f"Input Size: {input_size}")
print(f"Layer 1: {input_size} → 64 (ReLU + Dropout)")
print(f"Layer 2: 64 → 128 (ReLU + Dropout)")
print(f"Layer 3: 128 → 512 (ReLU + Dropout)")
print(f"Output Layer: 512 → 1 (Linear)")
print("="*60)

In [ ]:
# Define loss function and optimizer
criterion = nn.MSELoss()  # Mean Squared Error for regression
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss function: MSELoss")
print("Optimizer: Adam")
print(f"Learning rate: 0.001")

In [ ]:
# Training function
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_mae = 0.0
    
    for X_batch, y_batch in tqdm(train_loader, desc="Training"):
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device).unsqueeze(1)  # Add dimension for output
        
        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Calculate metrics
        total_loss += loss.item()
        mae = torch.mean(torch.abs(outputs - y_batch)).item()
        total_mae += mae
    
    avg_loss = total_loss / len(train_loader)
    avg_mae = total_mae / len(train_loader)
    
    return avg_loss, avg_mae


# Validation function
def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    
    with torch.no_grad():
        for X_batch, y_batch in tqdm(val_loader, desc="Validating"):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            total_loss += loss.item()
            mae = torch.mean(torch.abs(outputs - y_batch)).item()
            total_mae += mae
    
    avg_loss = total_loss / len(val_loader)
    avg_mae = total_mae / len(val_loader)
    
    return avg_loss, avg_mae


print("Training and validation functions defined!")

In [ ]:
# Train the model
num_epochs = 10
history = {
    'train_loss': [],
    'train_mae': [],
    'val_loss': [],
    'val_mae': []
}

for epoch in range(num_epochs):
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    
    train_loss, train_mae = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_mae = validate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['train_mae'].append(train_mae)
    history['val_loss'].append(val_loss)
    history['val_mae'].append(val_mae)
    
    print(f"Train Loss: {train_loss:.4f}, Train MAE: {train_mae:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val MAE: {val_mae:.4f}")

print("\nTraining completed!")

In [ ]:
# Plot training history
plt.figure(figsize=(14, 5))

# Loss plot
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Training Loss', marker='o')
plt.plot(history['val_loss'], label='Validation Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

# MAE plot
plt.subplot(1, 2, 2)
plt.plot(history['train_mae'], label='Training MAE', marker='o')
plt.plot(history['val_mae'], label='Validation MAE', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Mean Absolute Error')
plt.title('Training and Validation MAE')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
def evaluate_test(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    all_predictions = []
    all_actuals = []
    
    with torch.no_grad():
        for X_batch, y_batch in tqdm(test_loader, desc="Testing"):
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)
            
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            
            total_loss += loss.item()
            mae = torch.mean(torch.abs(outputs - y_batch)).item()
            total_mae += mae
            
            all_predictions.extend(outputs.cpu().numpy().flatten().tolist())
            all_actuals.extend(y_batch.cpu().numpy().flatten().tolist())
    
    avg_loss = total_loss / len(test_loader)
    avg_mae = total_mae / len(test_loader)
    
    return avg_loss, avg_mae, all_predictions, all_actuals


# Evaluate on test set
test_loss, test_mae, predictions, actuals = evaluate_test(model, test_loader, criterion, device)

print(f"\n{'='*60}")
print("TEST SET EVALUATION")
print(f"{'='*60}")
print(f"Test MSE Loss: {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}")
print(f"Number of test samples: {len(actuals)}")

In [ ]:
# Visualize predictions vs actuals
plt.figure(figsize=(12, 5))

# Predictions vs Actuals scatter plot
plt.subplot(1, 2, 1)
plt.scatter(actuals, predictions, alpha=0.5, s=20)
plt.plot([min(actuals), max(actuals)], [min(actuals), max(actuals)], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Salary')
plt.ylabel('Predicted Salary')
plt.title('Predictions vs Actual Salary')
plt.legend()
plt.grid(True)

# Residuals plot
plt.subplot(1, 2, 2)
residuals = [a - p for a, p in zip(actuals, predictions)]
plt.scatter(predictions, residuals, alpha=0.5, s=20)
plt.axhline(y=0, color='r', linestyle='--', lw=2)
plt.xlabel('Predicted Salary')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Function to make predictions on new data
def predict_salary(features, model, device):
    """
    Predict salary for given features
    
    Args:
        features: numpy array or pandas Series of shape (input_size,)
        model: trained model
        device: device to use
    
    Returns:
        predicted salary
    """
    if isinstance(features, pd.Series):
        features = features.values
    
    features_tensor = torch.from_numpy(features).float().unsqueeze(0).to(device)
    
    model.eval()
    with torch.no_grad():
        prediction = model(features_tensor)
    
    return prediction.item()


# Test prediction on a sample
print(f"\n{'='*60}")
print("SAMPLE PREDICTION")
print(f"{'='*60}")

sample_idx = 0
sample_features = X_test.iloc[sample_idx].values
sample_actual = y_test.iloc[sample_idx]
predicted_salary = predict_salary(sample_features, model, device)

print(f"Sample Index: {sample_idx}")
print(f"Actual Salary: ${sample_actual:.2f}")
print(f"Predicted Salary: ${predicted_salary:.2f}")
print(f"Difference: ${abs(sample_actual - predicted_salary):.2f}")

In [ ]:
# Save the model
model_save_path = 'salary_predictor_ann_pytorch.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

# Save the full model
model_full_path = 'salary_predictor_ann_pytorch_full.pt'
torch.save(model, model_full_path)
print(f"Full model saved to {model_full_path}")

# Save scaler for preprocessing new data
scaler_save_path = 'salary_predictor_scaler.pkl'
with open(scaler_save_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"Scaler saved to {scaler_save_path}")

# Save configuration
config = {
    'input_size': input_size,
    'test_loss': float(test_loss),
    'test_mae': float(test_mae),
    'dropout_rate': 0.5,
    'learning_rate': 0.001,
    'batch_size': batch_size,
    'num_epochs': num_epochs
}
config_save_path = 'salary_predictor_config.json'
with open(config_save_path, 'w') as f:
    json.dump(config, f, indent=4)
print(f"Config saved to {config_save_path}")

In [ ]:
# Function to load and use the saved model
def load_model_for_prediction(model_path, scaler_path, config_path, device):
    """
    Load saved model and scaler for inference
    """
    # Load configuration
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    # Load scaler
    with open(scaler_path, 'rb') as f:
        scaler = pickle.load(f)
    
    # Initialize and load model
    model = SalaryPredictorANN(config['input_size'])
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    
    return model, scaler, config


print(f"Model loading function defined!")
print(f"\nYou can load the saved model using:")
print(f"model, scaler, config = load_model_for_prediction('{model_save_path}', '{scaler_save_path}', '{config_save_path}', device)")

In [ ]:
# Summary of ANN Architecture
print(f"""
{'='*70}
ANN SALARY PREDICTOR - MODEL SUMMARY
{'='*70}

MODEL ARCHITECTURE:
- Input Layer: {input_size} features
- Hidden Layer 1: 64 neurons + ReLU + Dropout(0.5)
- Hidden Layer 2: 128 neurons + ReLU + Dropout(0.5)
- Hidden Layer 3: 512 neurons + ReLU + Dropout(0.5)
- Output Layer: 1 neuron + Linear (for regression)

TRAINING CONFIGURATION:
- Optimizer: Adam (lr=0.001)
- Loss Function: Mean Squared Error (MSE)
- Batch Size: {batch_size}
- Epochs: {num_epochs}
- Validation Split: 20%

TEST PERFORMANCE:
- Test MSE Loss: {test_loss:.4f}
- Test MAE: {test_mae:.4f}
- Total Test Samples: {len(actuals)}

FEATURES:
✓ One-hot encoding for categorical variables
✓ Ordinal encoding for education_level and company_size
✓ StandardScaler normalization
✓ Dropout regularization to prevent overfitting
✓ Train-validation-test split strategy
✓ Comprehensive evaluation metrics

{'='*70}
""")